Functors

A functor is simply a trait (or interface) that implements the map method.

Functors always have a map function
Functors can always be composed


Signature

trait Functor[F[_]] : 
	def map[A, B](fa: F[A])(f: A => B): F[B] 

In [ ]:
import cats.Functor

Functor[Option].map(Some(2))(_ + 1)

Functor[Option].map(None:Option[Int])(_ + 1)


In [ ]:
val funcOpt = Functor[Option]
val funcList = Functor[List]
val functCompose = (funcList compose funcOpt)

val list = List(3,6,4,2).map(Option(_))++List(None)

functCompose.map(list)(_ * 3)


In [ ]:
case class Evaluation[A](grade:A,remark:String)
given Functor[Evaluation] with
  def map[A,B](fa: Evaluation[A])(f: A => B) =             
      Evaluation(f(fa.grade),fa.remark)
val e1 = Evaluation(3.5,"not good")

import cats.syntax.functor._
e1.map(_.toInt)


In [ ]:
val count_a = (word:String) => word.count(_ == 'a')
val source = List("Avatar", "the", "last","airbender")
val product = Functor[List].fproduct(source)(count_a).toMap

product.get("Avatar").getOrElse(0) 


Monads

A monad is a trait (or interface) that implements two methods: 

flatMap and unit

Every monad is also a functor: must have a map function. 
We can have map combining flatmap and unit.


trait Monad[F[_]] : 
	def pure[A](a: A): F[A] 
	def flatMap[A, B](fa: F[A])(f: A => F[B]): F[B] 

In [ ]:
import cats.Monad

val anOpt = Monad[Option].pure(5)

val otherOpt = Monad[Option].flatMap(anOpt)(e => Option(e + 2))

val anotherOpt = Monad[Option].map(otherOpt)(e => 300 + e)


In [ ]:
import cats.Monad

val aList= Monad[List].pure(2)

val otherList = 
	Monad[List].flatMap(List(4,6,2))(e => List(e-1,e,e+1))


In [ ]:
Monad[Option].pure(25)

import cats.syntax.applicative._
25.pure[Option]


In [ ]:
enum Unit(val code:String) :
  case Kelvin extends Unit("K")
  case Celsius extends Unit("C")
  case Farenheit extends Unit("F")

case class Temperature[T](value:T,unit:Unit=Unit.Celsius):
  override def toString() = s"${value.toString}°${unit.code}"


In [ ]:
given Monad[Temperature] with
  def flatMap[A,B](fa:Temperature[A])(f:A => Temperature[B]):Temperature[B] =
    f(fa.value)
     
  def pure[A](x: A): Temperature[A] = Temperature(x)
  def tailRecM[A,B](a: A)(f: A => Temperature[Either[A, B]]):Temperature[B] =
	 ???

Monad[Temperature].pure(34)

34.pure[Temperature]


In [ ]:
val temp = Temperature(34)

Monad[Temperature].flatMap(temp)(t => Temperature(t+273.15,Unit.Kelvin))

import cats.syntax.monad._
import cats.syntax.flatMap._
import cats.syntax.functor._

temp.flatMap(e => Temperature(e+273.15,Unit.Kelvin))


In [ ]:
val optTemp = Monad[Option] compose Monad[Temperature]

optTemp.pure(None)

optTemp.pure(45)


In [ ]:
val either1:Either[String,Int] = Left("4")

val either2:Either[String,Int] = Right(4)


In [ ]:
import cats.syntax.either._

def computation[T](param:T):Either[String,Double] = param match
  case t:Int => (t*0.5).asRight
  case d:Double => (d*0.5).asRight
  case _ => "Not a good param".asLeft


computation("10")


In [ ]:
import cats.syntax.either._

3.asLeft

import cats.syntax.functor._
import cats.syntax.monad._
import cats.syntax.flatMap._

val f = computation(10).flatMap(a => a.asRight)


In [ ]:
val positive = (i:Int) => i.asRight[String].ensure("Should be positive")(_ > 0)

positive(-3)

positive(2)

123.asRight[String].swap


In [ ]:
Either.catchOnly[NumberFormatException]("123a".toInt)

Either.catchOnly[NumberFormatException](3/0)


In [ ]:
import cats.Foldable
import cats.instances.list
import cats.instances.seq
import cats.syntax.option._

val list = (1 to 10).toList

Foldable[List].foldLeft(list,0)(_ - _)

Foldable[List].fold(list)


In [ ]:
import cats.Foldable
import cats.instances.list
import cats.instances.seq
import cats.syntax.option._

val list = (1 to 10).toList

Foldable[List].foldMap(list)(_.toString)

Foldable[List].forall(list)(_ <= 3)


In [ ]:
import cats.syntax.foldable._

list.combineAll

list.foldMap(_.toString)

val comp = (Foldable[List] compose Foldable[Option])

comp.combineAll(List(4.some,5.some))
